# Dimensional — Fact Sales Table

Build `globalmart.gold.fact_sales` from **delivered** order items with FKs to all four dimensions. Validates row count, revenue, and non-null FKs.

In [ ]:
import sys
import importlib

notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
REPO_ROOT = notebook_path.split("/notebooks/")[0]
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import src.dimensional.fact_sales as fact_module
importlib.reload(fact_module)

from src.dimensional.fact_sales import (
    FactSalesConfig,
    build_fact_sales,
    validate_fact_sales,
    run_fact_sales,
)
from src.ingestion.idempotent_loader import save_run_report_to_volume

config = FactSalesConfig()
print("Loaded from:", fact_module.__file__)

In [ ]:
fact = build_fact_sales(spark, config)
fact.write.format("delta").mode("overwrite").saveAsTable(config.target_table)
print("Fact rows:", spark.table(config.target_table).count())
display(spark.table(config.target_table).limit(10))

In [ ]:
validation = validate_fact_sales(spark, spark.table(config.target_table), config)
print(validation)

In [ ]:
import json

report = run_fact_sales(spark, config)
print(json.dumps(report, indent=2, default=str))
try:
    save_run_report_to_volume(
        report,
        dbutils,
        "/Volumes/globalmart/metadata/run_reports/dimensional_fact_sales.json",
    )
except Exception as exc:
    print("Volume save skipped:", exc)